### this notebook is the start of the network wrangler process. 
this starts from scratch, with json and new rail links.





In [1]:
import os
import sys
import numpy as np

from pyproj import CRS
from pathlib import Path

from projectcard import read_cards

import network_wrangler
from network_wrangler import load_roadway
from network_wrangler import load_transit
from network_wrangler import create_scenario
from network_wrangler import Scenario
from network_wrangler.roadway import write_roadway
from network_wrangler.transit import write_transit

from met_council_wrangler import MetCouncil_Parameters
from met_council_wrangler import metcouncil_roadway
from met_council_wrangler import metcouncil_transit

from cube_wrangler import Parameters
from cube_wrangler import util
from cube_wrangler import roadway
from cube_wrangler import StandardTransit

In [2]:
network_wrangler.setup_logging()

In [3]:
%reload_ext autoreload
%autoreload 2

# remote i/o

In [4]:
input_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00")
cc_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00")
rail_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2")
tran_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2")

net_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks")
output_network = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\output")

metcouncil_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler")
cube_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler")


In [5]:
project_card_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1")
project_card_dir2 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections2_v1")
project_card_dir3 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMissingRoads_v1")
project_card_dir4 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseAttribute_v1")

In [6]:
metcouncil_parameters = MetCouncil_Parameters(
    metcouncil_wrangler_base_dir=metcouncil_wrangler_dir,
    cube_wrangler_base_dir=cube_wrangler_dir
)

2024-10-29 16:21:39, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 16:21:39, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler


# Load Version00

In [7]:
link_file = os.path.join(input_dir, 'standard_networks', 'links.json')
node_file = os.path.join(input_dir, 'standard_networks', 'nodes.geojson')
shape_file = os.path.join(input_dir, 'standard_networks', 'shapes.geojson')

roadway_net = load_roadway(
    links_file=link_file,
    nodes_file=node_file,
    shapes_file=shape_file,
)

2024-10-29 16:21:39, DEBUG: Reading nodes from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\standard_networks\nodes.geojson.
2024-10-29 16:21:39, DEBUG: Estimated read time: 3 seconds.
2024-10-29 16:21:55, DEBUG: Read 414131 nodes from file in 15.92.
2024-10-29 16:21:55, DEBUG: Turning node data into official nodes_df
2024-10-29 16:22:02, INFO: Read 414131 nodes from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\standard_networks\nodes.geojson in 22.71.
2024-10-29 16:22:03, INFO: Reading links from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\standard_networks\links.json.
2024-10-29 16:22:03, DEBUG: Estimated read time: 3 minutes.
2024-10-29 16:22:49, DEBUG: Read 1061566 links in 46.7.
2024-10-29 16:22:49, DEBUG: Creating 1061566 links.
2024-10-29 16:24:15, DEBUG: 1061566 new links.
2024-10-29 16:24:15, INFO: Rea

In [8]:
roadway_net.links_df.shape

(1061566, 52)

In [9]:
roadway_net.nodes_df.shape

(414131, 12)

In [10]:
transit_net = load_transit(os.path.join(tran_dir,"standard_transit_network"))

2024-10-29 16:24:25, INFO: Reading GTFS feed tables from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2\standard_transit_network
2024-10-29 16:24:25, DEBUG: ...reading Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2\standard_transit_network\frequencies.txt.
2024-10-29 16:24:25, DEBUG: ...reading Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2\standard_transit_network\routes.txt.
2024-10-29 16:24:25, DEBUG: ...reading Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2\standard_transit_network\shapes.txt.
2024-10-29 16:24:26, DEBUG: ...reading Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\v0.5.2\standard_transit_network\stops.txt.
2024-10-29 16:24:26, DEBUG: ...reading Z:\Met_Council\3

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\transit\io.py:81: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file)


2024-10-29 16:24:26, INFO: Initializing frequencies
2024-10-29 16:24:26, DEBUG: Validating + coercing value to frequencies
2024-10-29 16:24:26, DEBUG: PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
2024-10-29 16:24:26, INFO: Initializing routes
2024-10-29 16:24:26, DEBUG: Validating + coercing value to routes
2024-10-29 16:24:26, DEBUG: PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
2024-10-29 16:24:26, DEBUG: Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
2024-10-29 16:24:26, INFO: Initializing shapes
2024-10-29 16:24:26, DEBUG: Validating + coercing value to shapes
2024-10-29 16:24:26, INFO: Initializing stops
2024-10-29 16:24:26, DEBUG: Validating + coercing value to stops
2024-10-29 16:24:26, DEBUG: PK tabl

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\engines\pandas_engine.py:873: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  col = to_datetime_fn(col, **self.to_datetime_kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\engines\pandas_engine.py:873: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  col = to_datetime_fn(col, **self.to_datetime_kwargs)


2024-10-29 16:24:27, DEBUG: Creating new TransitNetwork.


In [11]:
roadway_net.links_df = roadway_net.links_df[roadway_net.links_df.A != roadway_net.links_df.B]

### Attribute the Network

In [12]:
roadway_net.links_df = roadway_net.links_df.drop('lanes', axis = 1)

In [13]:
# make sure the data types of the boolean columns are correct
roadway_net.links_df['bus_only'] = False

for c in list(set(roadway_net.links_df.columns) & set(metcouncil_parameters.bool_col)):
    roadway_net.links_df[c] = roadway_net.links_df[c].replace(
        {
            np.nan: False,
            "": False,
            "0": False,
            "1": True
        }
    )

In [14]:
r_net = metcouncil_roadway.calculate_number_of_lanes_from_reviewed_network(
    roadway_net=roadway_net,
    parameters=metcouncil_parameters,
)
r_net.links_df.lanes.value_counts()

2024-10-29 16:24:31, INFO: Calculating Number of Lanes as network variable: 'lanes'
2024-10-29 16:24:31, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 16:24:31, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
2024-10-29 16:24:31, DEBUG: Calculating Centroid Connectors
2024-10-29 16:24:31, INFO: Calculating Centroid Connector and adding as roadway network variable: centroidconnect
2024-10-29 16:24:31, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 16:24:31, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
2024-10-29 16:24:31, DEBUG: Calculating Centroid Connectors using highest TAZ number: 3100
2024-10-29 16:24:31, INFO: Finished calculating c

lanes
1.0    1026496
2.0      32020
3.0       2595
4.0        369
5.0         71
6.0          5
7.0          1
Name: count, dtype: int64

In [15]:
r_net.links_df.roadway.value_counts()

roadway
residential       420553
service           264058
footway           117014
tertiary          113519
cycleway           83868
secondary          36639
primary            14067
motorway_link       3604
trunk               3051
motorway            2757
secondary_link       839
trunk_link           601
tertiary_link        538
primary_link         449
Name: count, dtype: int64

In [16]:
r_net = metcouncil_roadway.calculate_assign_group_and_roadway_class_from_reviewed_network(
        roadway_net=r_net,
        parameters=metcouncil_parameters,
)
r_net.links_df.assign_group.value_counts(dropna=False)

2024-10-29 16:25:05, INFO: Calculating Assignment Group and Roadway Class as network variables: 'assign_group' and 'roadway_class'
2024-10-29 16:25:05, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 16:25:05, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
2024-10-29 16:25:05, DEBUG: Calculating Centroid Connectors
2024-10-29 16:25:05, INFO: Centroid Connector Variable 'centroidconnect' already in network. Returning without overwriting.
2024-10-29 16:26:44, INFO: Finished calculating assignment group variable assign_group and roadway class variable roadway_class


assign_group
50.0     294095
101.0    241490
103.0    204010
102.0    117014
7.0      101584
6.0       76802
5.0       15025
15.0       3045
3.0        2590
1.0        2455
4.0        2036
2.0        1258
11.0        153
Name: count, dtype: int64

In [17]:
r_net.links_df.roadway_class.value_counts()

roadway_class
101.0    562514
50.0     304018
40.0     124841
30.0      42279
20.0      20590
60.0       5416
10.0       1780
70.0        119
Name: count, dtype: int64

### Add Rail links and nodes

In [18]:
r_net = metcouncil_roadway.add_rail_links_and_nodes(
    roadway_network = r_net,
    parameters = metcouncil_parameters,
    rail_links_file = os.path.join(rail_dir, 'rail_links.geojson'),
    rail_nodes_file = os.path.join(rail_dir, 'rail_nodes.geojson'),
)

2024-10-29 16:26:45, INFO: Adding centroid and centroid connector to standard network
2024-10-29 16:26:46, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 16:26:46, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
2024-10-29 16:26:57, DEBUG: Reading shapes from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\standard_networks\shapes.geojson.
2024-10-29 16:26:57, DEBUG: Estimated read time: 8 seconds.
2024-10-29 16:27:24, DEBUG: Read 556541 shapes from file in 27.74.
2024-10-29 16:27:24, DEBUG: Creating 556541 shapes.
2024-10-29 16:27:25, INFO: Read 556541 shapes from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\standard_networks\shapes.geojson in 28.81.
2024-10-29 16:27:26, DEBUG: Creating 556636 shapes

In [19]:
# check if missing IDs
# rail nodes does not have osm IDs, they have shst IDs
# rail links does not have osm IDs, they have shst IDs

# if node missing shst id
print(r_net.nodes_df.shst_node_id.isnull().sum())
print(r_net.nodes_df.shst_node_id.nunique())
# if node missing osm id
print(r_net.nodes_df.osm_node_id.isnull().sum() + len(r_net.nodes_df[r_net.nodes_df.osm_node_id==""]))
# if node missing osm id
print(r_net.nodes_df.osm_node_id.dtype)
print(r_net.nodes_df[r_net.nodes_df.osm_node_id == 0])
print(r_net.nodes_df.osm_node_id.nunique())
# if node missing model node id
print(r_net.nodes_df.model_node_id.nunique())

# if link missing 
print(r_net.links_df.shstReferenceId.isnull().sum())
print(r_net.links_df.shstReferenceId.nunique())
print(r_net.links_df.model_link_id.nunique())

# if link missing node id
print(r_net.links_df.fromIntersectionId.isnull().sum())
print(r_net.links_df.toIntersectionId.isnull().sum())
print(r_net.links_df.u.isnull().sum())
print(r_net.links_df[r_net.links_df.u == 0])
print(r_net.links_df.v.isnull().sum())
print(r_net.links_df[r_net.links_df.v == 0])

0
414225
1998
object
Empty GeoDataFrame
Columns: [osm_node_id, shst_node_id, county, drive_access, walk_access, bike_access, model_node_id, rail_only, geometry, X, Y, projects]
Index: []
412228
414225
0
1061652
1061652
0
0
5077
Empty GeoDataFrame
Columns: [shstReferenceId, shape_id, shstGeometryId, fromIntersectionId, toIntersectionId, u, v, nodeIds, wayId, roadClass, oneWay, roundabout, link, oneway, lanes_osm, ref, name, highway, service, width, maxspeed, access, junction, bridge, tunnel, landuse, area, key, forward, backReferenceId, metadata, source, roadway, drive_access, walk_access, bike_access, county, length, A, B, model_link_id, locationReferences, rail_only, geometry, bus_only, distance, projects, managed, price, ML_projects, osm_link_id, centroidconnect, lanes, assign_group, roadway_class]
Index: []

[0 rows x 55 columns]
5077
Empty GeoDataFrame
Columns: [shstReferenceId, shape_id, shstGeometryId, fromIntersectionId, toIntersectionId, u, v, nodeIds, wayId, roadClass, oneWay,

# Create Scenario 00

In [20]:
# the new version of transit network (v0.5) is built on top of v01 roadway network (with all base project cards applied already)
# so exlcued transit network in the base_scenario here (otherwise it will fail because the roadway and transit are inconsistent)
# will add the transit network back in v01 scenario later
base_scenario = {"road_net": r_net}
# base_scenario = {"road_net": r_net, "transit_net": transit_net} 

In [21]:
version_00_scenario = create_scenario(base_scenario = base_scenario)

2024-10-29 16:27:30, INFO: Creating Scenario
2024-10-29 16:27:30, WARNING: Base_scenario doesn't contain ['road_net', 'transit_net', 'applied_projects', 'conflicts']


## Save version 00 standard networks

In [22]:
write_roadway(r_net, file_format="geojson", out_dir= os.path.join(net_dir, 'v00', 'standard_networks', 'roadway'), overwrite=True)
write_transit(transit_net, file_format="txt", out_dir= os.path.join(net_dir, 'v00', 'standard_networks', 'transit'), overwrite=True)

2024-10-29 16:27:34, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v00\standard_networks\roadway\link.json.
2024-10-29 16:27:59, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v00\standard_networks\roadway\node.geojson.
2024-10-29 16:28:14, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v00\standard_networks\roadway\shape.geojson.
2024-10-29 16:28:42, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v00\standard_networks\transit\frequencies.txt.
2024-10-29 16:28:42, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v00\standard_networks\transit\routes.txt.
2024-10-29 16:28:42, DEBUG: Writing to Z:

In [23]:
roadway_net.links_df.model_link_id.max()

1061661

In [24]:
roadway_net.nodes_df.shape

(414225, 12)

In [25]:
roadway_net.nodes_df.model_node_id.max()

417325

In [26]:
roadway_net.links_df.columns

Index(['shstReferenceId', 'shape_id', 'shstGeometryId', 'fromIntersectionId',
       'toIntersectionId', 'u', 'v', 'nodeIds', 'wayId', 'roadClass', 'oneWay',
       'roundabout', 'link', 'oneway', 'lanes_osm', 'ref', 'name', 'highway',
       'service', 'width', 'maxspeed', 'access', 'junction', 'bridge',
       'tunnel', 'landuse', 'area', 'key', 'forward', 'backReferenceId',
       'metadata', 'source', 'roadway', 'drive_access', 'walk_access',
       'bike_access', 'county', 'length', 'A', 'B', 'model_link_id',
       'locationReferences', 'rail_only', 'geometry', 'bus_only', 'distance',
       'projects', 'managed', 'price', 'ML_projects', 'osm_link_id',
       'centroidconnect', 'lanes', 'assign_group', 'roadway_class'],
      dtype='object')

# Create Scenario 00B

In [27]:
#this adds BaseAttribute, walk and bike variables to network, 
#manually calculates external connectors, transit priority, and some roadclass values
version_00b_scenario = create_scenario(
    base_scenario = version_00_scenario,
    project_card_filepath = project_card_dir4
)

2024-10-29 16:28:48, INFO: Creating Scenario
2024-10-29 16:28:50, INFO: Adding add walk and bike attributes to scenario.
2024-10-29 16:28:51, INFO: Adding year 2015 add centroid connector at external stations to scenario.
2024-10-29 16:28:51, DEBUG: Adding ['add walk and bike attributes'] to year 2015 add centroid connector at external stations dependency table.
2024-10-29 16:28:51, INFO: Adding manual changes to assignment group and roadway class to scenario.
2024-10-29 16:28:51, INFO: Adding add transit priority to scenario.
2024-10-29 16:28:51, INFO: Adding add transit priority part b to scenario.


In [28]:
version_00b_scenario.apply_all_projects()

2024-10-29 16:28:52, DEBUG: Ordered Projects: 
deque(['add transit priority part b', 'add transit priority', 'manual changes to assignment group and roadway class', 'add walk and bike attributes', 'year 2015 add centroid connector at external stations'])
2024-10-29 16:28:52, INFO: Applying add transit priority part b from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseAttribute_v1\TransitPriorityB.yml
2024-10-29 16:28:52, DEBUG: types: ['roadway_property_change']
2024-10-29 16:28:52, DEBUG: type: roadway_property_change
2024-10-29 16:28:52, DEBUG: - applying subproject: roadway_property_change
2024-10-29 16:28:52, DEBUG: Getting selection from key: 3abb6c6ef930af0e3c36733b0d78ea55f5c915f4
2024-10-29 16:28:52, DEBUG: Creating selection from selection dictionary: 
 {'links': {'model_link_id': [370236, 370317, 371158, 372095, 373213, 373307, 373466, 374150, 374272, 374414, 374500, 374722, 374883, 375525, 

<string>:30: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

<string>:70: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



2024-10-29 16:30:14, INFO: Applying add walk and bike attributes from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseAttribute_v1\add_bike_and_walk_attributes.wrangler
2024-10-29 16:30:14, DEBUG: types: ['pycode']
2024-10-29 16:30:14, DEBUG: type: pycode
2024-10-29 16:30:14, INFO: Applying Project to Roadway Network: add walk and bike attributes
2024-10-29 16:30:14, DEBUG: Applying calculated roadway project.
2024-10-29 16:30:16, INFO: Applying year 2015 add centroid connector at external stations from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseAttribute_v1\external_connectors.yaml
2024-10-29 16:30:16, DEBUG: types: ['roadway_addition']
2024-10-29 16:30:16, DEBUG: type: roadway_addition
2024-10-29 16:30:16, DEBUG: - applying subproject: roadway_addition
2024-10-29 16:30:17, DEBUG: Adding New Roadway Features: 
-Lin

# Create Scenario 00C

In [29]:
# this adds BaseCorrections, which correct attributes 
version_00c_scenario = create_scenario(
    base_scenario=version_00b_scenario,
    project_card_filepath = project_card_dir
)

2024-10-29 16:30:31, INFO: Creating Scenario
2024-10-29 16:30:40, INFO: Adding network cleanup 1 asgngrp 1 to scenario.
2024-10-29 16:30:41, INFO: Adding network cleanup 1 asgngrp 101 to scenario.
2024-10-29 16:30:41, INFO: Adding network cleanup 1 asgngrp 15 to scenario.
2024-10-29 16:30:41, INFO: Adding network cleanup 2 asgngrp 15 to scenario.
2024-10-29 16:30:41, INFO: Adding network cleanup 2 asgngrp 1 to scenario.
2024-10-29 16:30:42, INFO: Adding network cleanup 1 asgngrp 2 to scenario.
2024-10-29 16:30:42, INFO: Adding network cleanup 2 asgngrp 2 to scenario.
2024-10-29 16:30:42, INFO: Adding network cleanup 1 asgngrp 4 to scenario.
2024-10-29 16:30:42, INFO: Adding network cleanup 2 asgngrp 4 to scenario.
2024-10-29 16:30:42, INFO: Adding network cleanup 1 asgngrp 50 to scenario.
2024-10-29 16:30:43, INFO: Adding network cleanup 2 asgngrp 5 to scenario.
2024-10-29 16:30:43, INFO: Adding network cleanup 1 asgngrp 6 to scenario.
2024-10-29 16:30:43, INFO: Adding network cleanup 

In [30]:
version_00c_scenario.add_project_cards(
     list(
            read_cards(project_card_dir3).values()
        )
)

2024-10-29 16:30:58, INFO: Adding 12th_busramps to scenario.
2024-10-29 16:30:59, INFO: Adding anokaminorroads to scenario.
2024-10-29 16:30:59, INFO: Adding avts_split to scenario.
2024-10-29 16:30:59, INFO: Adding dtmpls_4thave to scenario.
2024-10-29 16:30:59, INFO: Adding fixwalkbike_warner to scenario.
2024-10-29 16:30:59, INFO: Adding hodgsonrd to scenario.
2024-10-29 16:31:00, INFO: Adding hodgsonrd2 to scenario.
2024-10-29 16:31:00, INFO: Adding lexington1 to scenario.
2024-10-29 16:31:00, INFO: Adding lex_lake_circlepines2 to scenario.
2024-10-29 16:31:00, INFO: Adding marq_second2 to scenario.
2024-10-29 16:31:01, INFO: Adding nicollet2 to scenario.
2024-10-29 16:31:01, INFO: Adding section1_columbus to scenario.
2024-10-29 16:31:01, INFO: Adding stillwaterblvd to scenario.
2024-10-29 16:31:01, INFO: Adding wacoutalink to scenario.


In [31]:
version_00c_scenario.apply_all_projects()

2024-10-29 16:31:02, DEBUG: Ordered Projects: 
deque(['wacoutalink', 'stillwaterblvd', 'section1_columbus', 'nicollet2', 'marq_second2', 'lex_lake_circlepines2', 'lexington1', 'hodgsonrd2', 'hodgsonrd', 'fixwalkbike_warner', 'dtmpls_4thave', 'avts_split', 'anokaminorroads', '12th_busramps', 'correct year 2018 assignment group and roadway class', 'correct year 2018 assignment group', 'route2_wash', 'nic_split', 'msp_deletes', 'missing 394 reverse', 'network cleanup 1 lanes 7', 'network cleanup 1 lanes 6', 'network cleanup 2 lanes 5 a', 'network cleanup 1 lanes 5', 'network cleanup 2 lanes 4 a', 'network cleanup 1 lanes 4', 'network cleanup 2 lanes 3 a', 'network cleanup 1 lanes 3', 'network cleanup 2 lanes 2g', 'network cleanup 2 lanes 2 a', 'network cleanup 1 lanes 2e', 'network cleanup 1 lanes 2 d3', 'network cleanup 1 lanes 2d part 2', 'network cleanup 1 lanes 2d', 'network cleanup 1 lanes 2 c3', 'network cleanup 1 lanes 2c part 2', 'network cleanup 1 lanes 2c', 'network cleanup 1 la

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\roadway\links\edit.py:274: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  links_df.loc[link_idx, prop_name] = prop_change.set


2024-10-29 16:32:04, DEBUG: - applying subproject: roadway_property_change
2024-10-29 16:32:04, DEBUG: Getting selection from key: 48a57d5ce3ffe862d4f27bed0fcc1c15f4b5fb01
2024-10-29 16:32:04, DEBUG: Creating selection from selection dictionary: 
 {'links': {'model_link_id': [584570, 378038], 'modes': ['any'], 'ignore_missing': True}}
2024-10-29 16:32:07, DEBUG: Created LinkSelection of type: query
2024-10-29 16:32:07, DEBUG: Applying roadway property change project.
2024-10-29 16:32:09, DEBUG: Initial link selection type: links.query
2024-10-29 16:32:09, INFO: Final selected links: 2
2024-10-29 16:32:09, DEBUG: 
                         shstReferenceId                          shape_id  \
378037  aee2925e19b86227098cd712f412b928  4cdb9bd5e519f98bfe5d0375fe30492e   
584569  0fbb4587ea8d8625c2551f48df503f3f  4cdb9bd5e519f98bfe5d0375fe30492e   

                          shstGeometryId                fromIntersectionId  \
378037  4cdb9bd5e519f98bfe5d0375fe30492e  9b6589fa35d6bb1551c719e2

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\roadway\links\edit.py:274: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  links_df.loc[link_idx, prop_name] = prop_change.set


2024-10-29 16:32:26, INFO: Applying marq_second2 from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMissingRoads_v1\marq_second.yml
2024-10-29 16:32:26, DEBUG: types: ['roadway_addition']
2024-10-29 16:32:26, DEBUG: type: roadway_addition
2024-10-29 16:32:26, DEBUG: - applying subproject: roadway_addition
2024-10-29 16:32:26, DEBUG: Adding New Roadway Features: 
-Links: 
[{'A': 9806, 'B': 343069, 'name': '2nd buslane', 'roadway': 'tertiary', 'drive_access': 1, 'walk_access': 1, 'bike_access': 1, 'county': '4', 'model_link_id': 1750030, 'rail_only': 0, 'bus_only': 1, 'centroidconnect': 0, 'assign_group': 98.0, 'roadway_class': 40.0, 'bike': 1, 'walk': 1, 'lanes': 2.0}, {'A': 343069, 'B': 343067, 'name': '2nd buslane', 'roadway': 'tertiary', 'drive_access': 1, 'walk_access': 1, 'bike_access': 1, 'county': '4', 'model_link_id': 1750031, 'rail_only': 0, 'bus_only': 1, 'centroidconnect': 0, 'assign_group':

...

2024-10-29 17:07:16, INFO: Applying bus routing correction cont from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\AsgnGrp98B.yml
2024-10-29 17:07:16, DEBUG: types: ['roadway_property_change']
2024-10-29 17:07:16, DEBUG: type: roadway_property_change
2024-10-29 17:07:16, DEBUG: - applying subproject: roadway_property_change
2024-10-29 17:07:16, DEBUG: Getting selection from key: b1634c27e2698ff2848f455bb3834f7b15d8af63
2024-10-29 17:07:16, DEBUG: Creating selection from selection dictionary: 
 {'links': {'model_link_id': [329664, 329776, 330407, 330478, 330485, 331295, 331817, 332133, 332441, 332528, 332711, 332730, 332802, 333903, 333972, 334476, 335234, 335251, 335983, 336296, 336810, 337208, 338719, 338749, 338791, 338839, 339238, 339690, 339872, 339991, 340321, 342631, 343266, 343602, 343641, 343644, 343814, 345133, 346715, 347953, 348254, 348519, 349215, 349695, 349749, 350083, 350

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\roadway\links\edit.py:274: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  links_df.loc[link_idx, prop_name] = prop_change.set


2024-10-29 17:07:38, INFO: Applying bus routing correction from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\AsgnGrp98.yml
2024-10-29 17:07:38, DEBUG: types: ['roadway_property_change']
2024-10-29 17:07:38, DEBUG: type: roadway_property_change
2024-10-29 17:07:38, DEBUG: - applying subproject: roadway_property_change
2024-10-29 17:07:38, DEBUG: Getting selection from key: b10cf4985d3c43db38a7e90786eae23009a1df42
2024-10-29 17:07:38, DEBUG: Creating selection from selection dictionary: 
 {'links': {'model_link_id': [957, 1035, 2174, 2419, 2763, 3835, 4662, 4819, 4867, 5636, 6672, 7415, 7914, 8428, 9386, 9452, 9616, 9632, 9738, 9775, 9893, 9960, 11387, 11973, 12448, 12638, 13086, 13451, 13580, 14182, 14496, 15327, 15695, 15743, 15755, 16116, 16402, 16720, 16894, 17072, 17122, 17174, 17240, 17558, 18089, 18520, 18623, 18886, 19136, 19420, 20394, 20419, 20507, 20563, 21844, 21875, 21987, 2

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\roadway\links\edit.py:274: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  links_df.loc[link_idx, prop_name] = prop_change.set


2024-10-29 17:08:00, INFO: Applying network cleanup 1 asgngrp 7 from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections_v1\AsgnGrp7.yml
2024-10-29 17:08:00, DEBUG: types: ['roadway_property_change']
2024-10-29 17:08:00, DEBUG: type: roadway_property_change
2024-10-29 17:08:00, DEBUG: - applying subproject: roadway_property_change
2024-10-29 17:08:00, DEBUG: Getting selection from key: 9c484204f2626dc3ceacba12abbcb568dace441d
2024-10-29 17:08:00, DEBUG: Creating selection from selection dictionary: 
 {'links': {'model_link_id': [327004, 309379, 47253, 80129, 9722, 443250, 243871, 77505, 244055, 488501], 'modes': ['any'], 'ignore_missing': True}}
2024-10-29 17:08:03, DEBUG: Created LinkSelection of type: query
2024-10-29 17:08:03, DEBUG: Applying roadway property change project.
2024-10-29 17:08:05, DEBUG: Initial link selection type: links.query
2024-10-29 17:08:05, INFO: Final selected links: 1

# Create Version 01

In [32]:
# this adds BaseCorrections2 , which are newer attribute clean up cards 
# there are too many dependencies to list so I'm running this seperately to make sure it applies last 
version_01_scenario = create_scenario(
    base_scenario=version_00c_scenario,
    project_card_filepath = project_card_dir2
)

2024-10-29 17:12:51, INFO: Creating Scenario
2024-10-29 17:12:54, INFO: Adding ag2cleanup to scenario.
2024-10-29 17:12:54, INFO: Adding ag2cleanup2 to scenario.
2024-10-29 17:12:54, INFO: Adding ag2cleanup3 to scenario.
2024-10-29 17:12:55, INFO: Adding ag2cleanup4 to scenario.
2024-10-29 17:12:55, INFO: Adding clean up 35w ramps to scenario.


In [33]:
version_01_scenario.apply_all_projects()

2024-10-29 17:12:55, DEBUG: Ordered Projects: 
deque(['clean up 35w ramps', 'ag2cleanup4', 'ag2cleanup3', 'ag2cleanup2', 'ag2cleanup'])
2024-10-29 17:12:55, INFO: Applying clean up 35w ramps from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections2_v1\DTRamps.yml
2024-10-29 17:12:55, DEBUG: types: ['roadway_property_change']
2024-10-29 17:12:55, DEBUG: type: roadway_property_change
2024-10-29 17:12:55, DEBUG: - applying subproject: roadway_property_change
2024-10-29 17:12:56, DEBUG: Getting selection from key: d894587f49f4f343b163f7f832eee5de8279d58f
2024-10-29 17:12:56, DEBUG: Creating selection from selection dictionary: 
 {'links': {'model_link_id': [432771, 293109, 43439, 175885, 534085, 146132], 'modes': ['any'], 'ignore_missing': True}}
2024-10-29 17:12:58, DEBUG: Created LinkSelection of type: query
2024-10-29 17:12:58, DEBUG: Applying roadway property change project.
2024-10-29 17:13:00,

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\roadway\links\edit.py:274: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  links_df.loc[link_idx, prop_name] = prop_change.set
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\roadway\links\edit.py:274: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  links_df.loc[link_idx, prop_name] = prop_change.set
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\network_wrangler\roadway\links\edit.py:274: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incom

2024-10-29 17:13:16, INFO: Applying ag2cleanup4 from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseCorrections2_v1\AG2Cleanup4.yml
2024-10-29 17:13:16, DEBUG: types: ['roadway_deletion', 'roadway_addition', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change', 'roadway_property_change']
2024-10-29 17:13:16, DEBUG: type: multiple
2024-10-29 17:13:16, DEBUG: - applying subproject: roadway_deletion
2024-10-29 17:13:16, DEBUG: Deleting Roadway Features: 
links=SelectLinksDict(all=False, name=None, ref=None, osm_link_id=None, model_link_id=[71064, 425702, 102753

In [34]:
version_01_scenario.applied_projects

['add transit priority part b',
 'add transit priority',
 'manual changes to assignment group and roadway class',
 'add walk and bike attributes',
 'year 2015 add centroid connector at external stations',
 'wacoutalink',
 'stillwaterblvd',
 'section1_columbus',
 'nicollet2',
 'marq_second2',
 'lex_lake_circlepines2',
 'lexington1',
 'hodgsonrd2',
 'hodgsonrd',
 'fixwalkbike_warner',
 'dtmpls_4thave',
 'avts_split',
 'anokaminorroads',
 '12th_busramps',
 'correct year 2018 assignment group and roadway class',
 'correct year 2018 assignment group',
 'route2_wash',
 'nic_split',
 'msp_deletes',
 'missing 394 reverse',
 'network cleanup 1 lanes 7',
 'network cleanup 1 lanes 6',
 'network cleanup 2 lanes 5 a',
 'network cleanup 1 lanes 5',
 'network cleanup 2 lanes 4 a',
 'network cleanup 1 lanes 4',
 'network cleanup 2 lanes 3 a',
 'network cleanup 1 lanes 3',
 'network cleanup 2 lanes 2g',
 'network cleanup 2 lanes 2 a',
 'network cleanup 1 lanes 2e',
 'network cleanup 1 lanes 2 d3',
 'ne

In [35]:
version_01_scenario.road_net.links_df['access'] = version_01_scenario.road_net.links_df['access'].fillna('')
version_01_scenario.road_net.links_df['access'] = version_01_scenario.road_net.links_df['access'].apply(
    lambda x: util.shorten_name(x)
)

In [36]:
version_01_scenario.road_net.links_df['name'] = version_01_scenario.road_net.links_df['name'].fillna('')
version_01_scenario.road_net.links_df['name'] = version_01_scenario.road_net.links_df['name'].apply(
    lambda x: util.shorten_name(x)
)

# Save version 01 standard networks

In [37]:
#i need to do this again once i get all the missing roads added


In [38]:
# add transit network to version 01 scenario
version_01_scenario.transit_net = transit_net

In [39]:
version_01_scenario.write(
    os.path.join(net_dir, 'v01', 'standard_networks'),
    name = 'v01',
    roadway_file_format = "geojson",
    transit_file_format = "txt",
    roadway_write = True,
    transit_write = True,
    projects_write = True,
    overwrite = True,
    roadway_convert_complex_link_properties_to_single_field=True
)

2024-10-29 17:27:58, WARNING: No complex properties detected to convert in links_df.                               Returning links_df as-is.
2024-10-29 17:27:59, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v01\standard_networks\roadway\v01_link.json.
2024-10-29 17:28:25, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v01\standard_networks\roadway\v01_node.geojson.
2024-10-29 17:28:41, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v01\standard_networks\roadway\v01_shape.geojson.
2024-10-29 17:29:04, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v01\standard_networks\transit\v01_frequencies.txt.
2024-10-29 17:29:04, DEBUG: Writing to Z:\Met_Council\31000743A\Task 1 Con

WindowsPath('Z:/Met_Council/31000743A/Task 1 Consolidate Architecture/consolidate_architecture/Networks/standard_networks/v01/standard_networks/v01_scenario.yml')

## have not updated below this mark - rarely need to export this base
## continue to export
## otherwise continue to notebook 02


# Make Travel Model Network

### Add centroid and centroid connectors

In [40]:
r_net = metcouncil_roadway.add_centroid_and_centroid_connector(
    roadway_network = version_01_scenario.road_net,
    parameters = metcouncil_parameters,
    centroid_file = os.path.join(input_dir, 'standard_networks', 'centroid_node.pickle'),
    centroid_connector_link_file = os.path.join(input_dir, 'standard_networks', 'cc_link.pickle'),
    centroid_connector_shape_file = os.path.join(input_dir, 'standard_networks', 'cc_shape.pickle'),
)

2024-10-29 17:29:31, INFO: Adding centroid and centroid connector to standard network
2024-10-29 17:29:31, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 17:29:31, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler


c:\Users\USYS671257\.conda\envs\wrangler\lib\pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again; shapely 2.1 will not have this compatibility.
  setstate(state)
c:\Users\USYS671257\.conda\envs\wrangler\lib\pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again; shapely 2.1 will not have this compatibility.
  setstate(state)
c:\Users\USYS671257\.conda\envs\wrangler\lib\pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again; shapely 2.1 will not have this compatibility.
  setstate(state)
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_roadway.py:2088: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt

2024-10-29 17:29:34, DEBUG: Creating 577540 shapes.
2024-10-29 17:29:35, INFO: Finished adding centroid and centroid connectors


### Add Rail access and egress links

In [41]:
r_net = metcouncil_roadway.add_rail_ae_connections(
    r_net,
    metcouncil_parameters,
    exclude_rail_node_id = [417275, 417255]
)

2024-10-29 17:29:36, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 17:29:36, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
2024-10-29 17:29:36, INFO: Creating rail access and egress connection links
2024-10-29 17:29:36, INFO: Exclude rail node id: [417275, 417255]
2024-10-29 17:30:14, DEBUG: Creating 577724 shapes.


In [42]:
m_net = metcouncil_roadway.roadway_standard_to_met_council_network(
    r_net,
    metcouncil_parameters    
)

2024-10-29 17:30:16, INFO: Renaming roadway attributes to be consistent with what metcouncil's model is expecting
2024-10-29 17:30:16, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 17:30:16, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
2024-10-29 17:30:16, INFO: Distance Variable 'distance' already in network. Returning without overwriting.
2024-10-29 17:30:16, INFO: Finished creating ML lanes variable: ML_lanes
2024-10-29 17:30:16, INFO: Finished creating hov corridor variable: segment_id
2024-10-29 17:30:16, INFO: Managed Variable 'managed' already in network. Returning without overwriting.
2024-10-29 17:30:16, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 17:30:16, INFO: MetCouncil Wrangler base directory set as: Z:\Met_

\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_roadway.py:284: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids_gdf["geometry"] = centroids_gdf["geometry"].centroid


2024-10-29 17:30:19, DEBUG: Reading Area Type Shapefile Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler\metcouncil_data\area_type\ThriveMSP2040CommunityDesignation.shp
2024-10-29 17:30:28, DEBUG: Area Type Codes Used: {23: 4, 24: 3, 25: 2, 35: 2, 36: 1, 41: 1, 51: 1, 52: 1, 53: 1, 60: 1}
2024-10-29 17:30:29, DEBUG: Downtown Area Type used boundary file: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler\metcouncil_data\area_type\downtownzones_TAZ.shp
2024-10-29 17:30:29, INFO: Finished Calculating Area Type from Spatial Data into variable: area_type
2024-10-29 17:30:30, INFO: Overwriting existing County Variable 'county' already in network
2024-10-29 17:30:30, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 17:30:30, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\softwa

\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_roadway.py:442: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids_gdf["geometry"] = centroids_gdf["geometry"].centroid
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_roadway.py:453: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:4326
Right CRS: EPSG:4269

  joined_gdf = gpd.sjoin(


2024-10-29 17:30:39, INFO: Finished Calculating county variable: county
2024-10-29 17:30:40, INFO: Calculating MPO as roadway network variable: mpo
2024-10-29 17:30:40, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 17:30:40, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
2024-10-29 17:30:40, DEBUG: MPO Counties: [,1,,, ,3,,, ,4,,, ,5,,, ,6,,, ,7,,, ,2,]
2024-10-29 17:30:40, INFO: Finished calculating MPO variable: mpo
2024-10-29 17:30:40, INFO: Adding Counts
2024-10-29 17:30:40, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 17:30:40, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
2024-10-29 17:30:40, DEBUG: Adding MNDOT Counts using 
- sh

...

2024-10-29 17:30:50, INFO: Added variable: AADT using Shared Streets Reference


\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\cube_wrangler\cube_wrangler\roadway.py:556: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  roadway_net.links_df[network_variable].fillna(0, inplace=True)


2024-10-29 17:30:50, INFO: Finished adding counts variable: AADT
2024-10-29 17:30:50, INFO: Filling nan for network from network wrangler


\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\cube_wrangler\cube_wrangler\roadway.py:666: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  roadway_net.links_df[x].fillna(0, inplace=True)
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\cube_wrangler\cube_wrangler\roadway.py:666: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result

2024-10-29 17:31:05, INFO: Splitting variables by time period and category
2024-10-29 17:31:17, DEBUG: No scoped values trn_priority. Returning default.
2024-10-29 17:31:30, DEBUG: No scoped values trn_priority. Returning default.
2024-10-29 17:31:41, DEBUG: No scoped values trn_priority. Returning default.
2024-10-29 17:31:53, DEBUG: No scoped values trn_priority. Returning default.
2024-10-29 17:32:05, DEBUG: No scoped values trn_priority. Returning default.
2024-10-29 17:32:05, WARNING: Specified variable to split: ttime_assert not in network variables: Index(['shstReferenceId', 'shape_id', 'shstGeometryId', 'fromIntersectionId',
       'toIntersectionId', 'u', 'v', 'nodeIds', 'wayId', 'roadClass', 'oneWay',
       'roundabout', 'link', 'oneway', 'lanes_osm', 'ref', 'name', 'highway',
       'service', 'width', 'maxspeed', 'access', 'junction', 'bridge',
       'tunnel', 'landuse', 'area', 'key', 'forward', 'backReferenceId',
       'metadata', 'source', 'roadway', 'drive_access', '

In [43]:
import geopandas as gpd
print(gpd.__version__)

1.0.1


In [44]:
# check if missing IDs
# centroids does not have osm and shst IDs
# centroid connectors does not have osm and shst IDs

# if node missing shst id
print(m_net.nodes_df.shst_node_id.isnull().sum())
print(m_net.nodes_df.shst_node_id.nunique())

# if node missing model node id
print(m_net.nodes_df.model_node_id.nunique())

# if link missing 
print(m_net.links_df.shstReferenceId.isnull().sum() + len(m_net.links_df[m_net.links_df.shstReferenceId==""]))
print(m_net.links_df.shstReferenceId.nunique())
print(m_net.links_df.model_link_id.nunique())

# if link missing node id
print(m_net.links_df.fromIntersectionId.isnull().sum())
print(m_net.links_df.toIntersectionId.isnull().sum())

0
414221
417312
42248
1061504
1103752
21342
21342


In [45]:
m_net.nodes_df.shape

(417312, 13)

In [46]:
m_net.nodes_df.columns

Index(['osm_node_id', 'shst_node_id', 'drive_access', 'walk_access',
       'bike_access', 'model_node_id', 'rail_only', 'geometry', 'X', 'Y',
       'projects', 'county', 'N'],
      dtype='object')

In [47]:
m_net.links_df.columns

Index(['shstReferenceId', 'shape_id', 'shstGeometryId', 'fromIntersectionId',
       'toIntersectionId', 'u', 'v', 'nodeIds', 'wayId', 'roadClass',
       ...
       'price_sov_NT', 'price_hov2_NT', 'price_hov3_NT', 'price_truck_NT',
       'access_EA', 'access_AM', 'access_MD', 'access_PM', 'access_NT',
       'geometry'],
      dtype='object', length=116)

# Write model network as shapefile

In [48]:
#trying to export all of them 
#out_cols = ['model_link_id', 'id', 'assign_group', 'drive_access', 'roadway_class',
#            'lanes_AM', 'lanes_MD', 'lanes_PM', 'lanes_NT', 'segment_id', 'HOV', 'bike', 'walk','roadway',
#            'price_sov_AM', 'geometry', 'managed']

roadway.write_roadway_as_shp(
    roadway_net = m_net,
    parameters = metcouncil_parameters,
    output_link_shp = os.path.join(output_network, 'fullnet_v01', 'shapefile', 'links_v01.shp'),
    output_node_shp = os.path.join(output_network, 'fullnet_v01', 'shapefile', 'nodes_v01.shp'),
    #link_output_variables = out_cols,
    data_to_csv = False,
    data_to_dbf = True,
    export_drive_only = False, # if user only wants drive links/nodes in the shapefile
)

2024-10-29 17:38:42, INFO: Writing Network as Shapefile
2024-10-29 17:38:42, DEBUG: Output Variables: 
 - model_link_id
 - link_id
 - A
 - B
 - shstGeometryId
 - shape_id
 - distance
 - roadway
 - name
 - roadway_class
 - roadway_class_idx
 - assign_group
 - mpo
 - AADT
 - area_type
 - county
 - centroidconnect
 - segment_id
 - drive_access
 - walk_access
 - bike_access
 - truck_access
 - bus_only
 - rail_only
 - bike_facility
 - bike
 - walk
 - MNPASS_CODE
 - MNPASS_PAY
 - managed
 - mrcc_id
 - ROUTE_SYS
 - trn_priority_AM
 - trn_priority_MD
 - trn_priority_PM
 - trn_priority_NT
 - ttime_assert_AM
 - ttime_assert_MD
 - ttime_assert_PM
 - ttime_assert_NT
 - lanes_AM
 - lanes_MD
 - lanes_PM
 - lanes_NT
 - access_AM
 - access_MD
 - access_PM
 - access_NT
 - count_year
 - count_AM
 - count_MD
 - count_PM
 - count_NT
 - count_daily
 - ML_lanes_AM
 - ML_lanes_MD
 - ML_lanes_PM
 - ML_lanes_NT
 - N
 - osm_node_id
 - bike_node
 - transit_node
 - walk_node
 - drive_node
 - geometry
 - X
 - Y
20

\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\cube_wrangler\cube_wrangler\roadway.py:876: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  links_dbf_df.to_file(output_link_shp)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'MNPASS_CODE' to 'MNPASS_COD'
  ogr_write(


# Write model network for Cube

In [49]:
roadway.write_roadway_as_fixedwidth(
    roadway_net = m_net,
    parameters = metcouncil_parameters,
    zones = metcouncil_parameters.zones,
    output_link_txt = os.path.join(output_network, 'fullnet_v01', 'links.txt'),
    output_node_txt = os.path.join(output_network,  'fullnet_v01','nodes.txt'),
    output_link_header_width_txt = os.path.join(output_network,  'fullnet_v01', 'links_header_width.txt'),
    output_node_header_width_txt = os.path.join(output_network,  'fullnet_v01','nodes_header_width.txt'),
    output_cube_network_script = os.path.join(output_network,  'fullnet_v01',  'make_complete_network_from_fixed_width_file.s'),
)

2024-10-29 17:40:21, DEBUG: Network Link Variables: 
 - shstReferenceId
 - shape_id
 - shstGeometryId
 - fromIntersectionId
 - toIntersectionId
 - u
 - v
 - nodeIds
 - wayId
 - roadClass
 - oneWay
 - roundabout
 - link
 - oneway
 - lanes_osm
 - ref
 - name
 - highway
 - service
 - width
 - maxspeed
 - access
 - junction
 - bridge
 - tunnel
 - landuse
 - area
 - key
 - forward
 - backReferenceId
 - metadata
 - source
 - roadway
 - drive_access
 - walk_access
 - bike_access
 - county
 - length
 - A
 - B
 - model_link_id
 - locationReferences
 - rail_only
 - bus_only
 - distance
 - projects
 - managed
 - price
 - ML_projects
 - osm_link_id
 - centroidconnect
 - lanes
 - assign_group
 - roadway_class
 - trn_priority
 - area_type
 - ramp_flag
 - bike
 - walk
 - MNPASS_CODE
 - MNPASS_PAY
 - segment_id
 - mpo
 - AADT
 - count_AM
 - count_MD
 - count_PM
 - count_NT
 - count_daily
 - count_year
 - trn_priority_EA
 - trn_priority_AM
 - trn_priority_MD
 - trn_priority_PM
 - trn_priority_NT
 - tti

In [50]:
version_01_scenario.transit_net.road_net = version_01_scenario.road_net
standard_transit_net = StandardTransit.fromTransitNetwork(version_01_scenario.transit_net, parameters=metcouncil_parameters)

In [51]:
standard_transit_net = metcouncil_transit.transit_standard_to_met_council_transit_network(
    transit_net = standard_transit_net,
    parameters = metcouncil_parameters,
    line_name_xwalk = os.path.join(output_network, 'fullnet_v01', "line_name_xwalk.csv")
)    

2024-10-29 17:52:11, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 17:52:11, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
No missing values found in column 'agency_raw_name'.
No missing values found in column 'agency_raw_name'.
No missing values found in column 'agency_raw_name'.
No missing values found in column 'agency_raw_name'.
No missing values found in column 'agency_raw_name'.
2024-10-29 17:52:13, INFO: Converting GTFS Standard Properties to MetCouncil's Cube Standard
2024-10-29 17:52:13, INFO: cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
2024-10-29 17:52:13, INFO: MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler


\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_transit.py:1308: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trip_df.groupby(["agency_id", "route_id", "direction_id", "shp_index"])
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_transit.py:1320: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trip_df.groupby(["agency_id", "route_id", "direction_id", "shp_index"])


2024-10-29 17:52:13, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:14, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:14, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:14, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:15, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:15, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:16, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:16, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:16, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:17, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:17, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:17, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:18, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:18, DEBUG: Validating + coercing value to shapes
2024-10-29 17:52:19, DEBUG: Validating + coercing value to shapes
2024-10-29

In [52]:
standard_transit_net.write_as_cube_lin(outpath = os.path.join(output_network, 'fullnet_v01', "transit.lin"))

In [53]:
version_01_scenario.road_net.nodes_df.model_node_id.max()

896904

In [54]:
version_01_scenario.road_net.nodes_df.model_node_id.shape

(417312,)

In [55]:
m_net.nodes_df.model_node_id.max()

896904

In [56]:
m_net.nodes_df.model_node_id.shape

(417312,)